# Hybrid Movie Recommendation System — with a Proactive Interface Variant

**Individual Project — Recommendation Systems**

This notebook implements a movie recommendation system in three layers:

1. **Content-based filtering** — recommends movies similar to ones a user already liked, based on genre overlap (TF-IDF + cosine similarity).
2. **Collaborative filtering** — recommends movies liked by *other users with similar taste*, using a user–item ratings matrix and cosine similarity.
3. **Hybrid scoring** — blends the two so the system works even for users with very few ratings (a classic "cold start" problem for pure collaborative filtering).

## The interface variation

Most recommender demos use the same pattern: the user *types a query* ("recommend me something like Inception") or picks an ID from a dropdown, and the system *reacts*. That's a **pull** interface.

This notebook instead implements two interfaces side by side:

- **Pull interface** — an interactive widget (dropdown + button) where the user actively asks for recommendations. This is the conventional pattern.
- **Push interface** — a "Daily Suggestion" cell that runs with *no user input at all*. It looks at the user's existing profile and proactively surfaces a recommendation, the way a proactive assistant would rather than waiting to be asked. It also shows a **confidence score**, which matters a lot for push-style systems (see the reflection at the end).

Run the cells top to bottom. Everything is self-contained — no external downloads, no API keys, no internet access needed. It runs the same in Google Colab or locally.


## 1. Setup

In [ ]:
# If a package is missing in your environment, uncomment the next line:
# !pip install pandas numpy scikit-learn ipywidgets -q

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import random

pd.set_option("display.max_colwidth", None)


## 2. Load the data

The `data/` folder contains two CSVs generated for this project:

- `movies.csv` — 30 fictional movies with a `movieId`, `title`, and pipe-separated `genres`.
- `ratings.csv` — synthetic 1–5 star ratings from 25 simulated users, built so users have realistic hidden "taste clusters" (each user secretly favors 2 genres), which gives the collaborative filter something real to find.

Using a small synthetic dataset instead of downloading MovieLens keeps the whole project reproducible with zero internet dependency — swap in the real MovieLens `ml-latest-small` CSVs later with the same column names and everything below still works unchanged.


In [ ]:
movies = pd.read_csv("data/movies.csv")
ratings = pd.read_csv("data/ratings.csv")

print(f"{len(movies)} movies, {len(ratings)} ratings, {ratings.userId.nunique()} users")
movies.head()


## 3. Content-based filtering (genre similarity)

In [ ]:
# TF-IDF over genre strings turns each movie's genre list into a vector.
# Movies with overlapping genres end up with high cosine similarity.

tfidf = TfidfVectorizer(token_pattern=r"[^|]+")
genre_matrix = tfidf.fit_transform(movies["genres"])
content_sim = cosine_similarity(genre_matrix)

content_sim_df = pd.DataFrame(content_sim, index=movies.movieId, columns=movies.movieId)

def content_based_recommend(movie_id, top_n=5):
    scores = content_sim_df[movie_id].drop(movie_id).sort_values(ascending=False)
    top_ids = scores.head(top_n).index
    result = movies.set_index("movieId").loc[top_ids].copy()
    result["similarity"] = scores.head(top_n).values
    return result.reset_index()

# quick sanity check
sample_id = movies.movieId.iloc[0]
print(f"Because you liked: {movies.loc[movies.movieId==sample_id, 'title'].values[0]}")
content_based_recommend(sample_id)


## 4. Collaborative filtering (user–user similarity)

In [ ]:
# Build a user x movie ratings matrix (missing values = 0, i.e. "not rated")
user_item = ratings.pivot_table(index="userId", columns="movieId", values="rating").fillna(0)

user_sim = cosine_similarity(user_item)
user_sim_df = pd.DataFrame(user_sim, index=user_item.index, columns=user_item.index)

def collaborative_recommend(user_id, top_n=5, k_neighbors=5):
    if user_id not in user_sim_df.index:
        return pd.DataFrame(columns=["movieId","title","genres","predicted_rating"])

    # Find the k most similar users (excluding the user themselves)
    neighbors = user_sim_df[user_id].drop(user_id).sort_values(ascending=False).head(k_neighbors)

    already_rated = set(ratings.loc[ratings.userId==user_id, "movieId"])
    candidate_scores = {}

    for neighbor_id, sim in neighbors.items():
        if sim <= 0:
            continue
        neighbor_ratings = ratings[ratings.userId==neighbor_id]
        for _, row in neighbor_ratings.iterrows():
            if row.movieId in already_rated:
                continue
            candidate_scores.setdefault(row.movieId, [0,0])
            candidate_scores[row.movieId][0] += sim * row.rating
            candidate_scores[row.movieId][1] += sim

    predicted = {
        mid: num/den for mid, (num, den) in candidate_scores.items() if den > 0
    }
    top_ids = sorted(predicted, key=predicted.get, reverse=True)[:top_n]

    result = movies.set_index("movieId").loc[top_ids].copy()
    result["predicted_rating"] = [round(predicted[i], 2) for i in top_ids]
    return result.reset_index()

collaborative_recommend(user_id=1)


## 5. Hybrid recommender

Pure collaborative filtering struggles for users with very few ratings ("cold start").
The hybrid blends both scores so the system degrades gracefully instead of failing outright.


In [ ]:
def hybrid_recommend(user_id, top_n=5, content_weight=0.4, collab_weight=0.6):
    already_rated = ratings.loc[ratings.userId==user_id, "movieId"]
    if len(already_rated) == 0:
        return pd.DataFrame(columns=["movieId","title","genres","score"])

    # Content signal: average similarity to everything the user rated highly (>=4)
    liked = ratings[(ratings.userId==user_id) & (ratings.rating>=4)]["movieId"]
    if len(liked) > 0:
        content_scores = content_sim_df[liked].mean(axis=1)
    else:
        content_scores = pd.Series(0, index=movies.movieId)

    # Collaborative signal
    collab_df = collaborative_recommend(user_id, top_n=len(movies))
    collab_scores = pd.Series(0.0, index=movies.movieId)
    if not collab_df.empty:
        collab_scores.update(collab_df.set_index("movieId")["predicted_rating"] / 5.0)

    combined = content_weight*content_scores.reindex(movies.movieId, fill_value=0) \
             + collab_weight*collab_scores.reindex(movies.movieId, fill_value=0)
    combined = combined.drop(index=set(already_rated), errors="ignore")

    top_ids = combined.sort_values(ascending=False).head(top_n).index
    result = movies.set_index("movieId").loc[top_ids].copy()
    result["score"] = combined.loc[top_ids].round(3).values
    return result.reset_index()

hybrid_recommend(user_id=1)


## 6. Interface #1 — Pull (the conventional pattern)

The user actively selects a profile and clicks a button. This is the "typing a question into a virtual assistant" model — the system does nothing until asked.


In [ ]:
user_dropdown = widgets.Dropdown(
    options=sorted(user_item.index.tolist()),
    description="User:",
    style={"description_width": "initial"}
)
get_recs_button = widgets.Button(description="Get my recommendations", button_style="primary")
output_area = widgets.Output()

def on_click(b):
    with output_area:
        clear_output()
        uid = user_dropdown.value
        rated_titles = movies[movies.movieId.isin(ratings[ratings.userId==uid].movieId)]["title"].tolist()
        print(f"User {uid} has rated: {', '.join(rated_titles[:5])}{'...' if len(rated_titles) > 5 else ''}\n")
        display(hybrid_recommend(uid))

get_recs_button.on_click(on_click)
display(widgets.HBox([user_dropdown, get_recs_button]), output_area)


## 7. Interface #2 — Push (the proactive variant)

This cell takes **no input from the user at all**. It picks up wherever the user's profile currently stands and proactively surfaces one suggestion, along with a confidence score — the way a genuinely proactive assistant would notify you rather than wait to be asked.

Notice the design choice: a push interface without a visible confidence signal is just noise. Showing the score is what makes proactive delivery trustworthy instead of annoying.


In [ ]:
def daily_suggestion(user_id):
    recs = hybrid_recommend(user_id, top_n=1)
    if recs.empty:
        print(f"Not enough data yet to proactively suggest anything for user {user_id}.")
        return
    row = recs.iloc[0]
    confidence = "High" if row.score > 0.5 else ("Medium" if row.score > 0.25 else "Low")
    display(HTML(f"""
    <div style="border:1px solid #ddd; border-radius:10px; padding:14px 18px; max-width:420px; font-family:sans-serif;">
      <div style="font-size:12px; color:#888; text-transform:uppercase; letter-spacing:0.05em;">Today's suggestion for User {user_id}</div>
      <div style="font-size:18px; font-weight:600; margin:6px 0;">{row.title}</div>
      <div style="color:#555; font-size:13px; margin-bottom:8px;">{row.genres}</div>
      <div style="font-size:12px; color:#888;">Confidence: <b>{confidence}</b> (score {row.score})</div>
    </div>
    """))

# Simulate the system proactively greeting a random user, unprompted
daily_suggestion(random.choice(user_item.index.tolist()))


## 8. Reflection — Will we always be typing questions into virtual assistants?

**Will interaction with intelligent systems keep improving?** Almost certainly — but the more interesting change isn't just "better answers to the same question." It's a shift in *who initiates the interaction*.

**Will we always be typing?** I don't think so, and this project is a small demonstration of why. Section 6 (the dropdown + button) is the interaction pattern we're all used to: the user frames a request, the system reacts. Section 7 (the daily suggestion) needs zero typing — the system already has enough context (the user's rating history) to act first.

A few forces are pushing interaction away from the query box:

- **Ambient context.** Systems increasingly have standing access to relevant signals — history, calendar, location, prior behavior — the ingredients needed to act without being asked. My hybrid recommender didn't need a typed query; it just needed to know who was asking.
- **Agentic execution.** Instead of answering a question, a system can just take the action (queue up the movie, reorder the part, send the reminder) and report what it did. That's a different interaction shape than Q&A entirely.
- **Interruption cost.** Typing has real friction — which is partly why chat interfaces stayed dominant even as voice and other modalities matured. A proactive system has to be right often enough that the interruption is worth it, or people tune it out, the way we ignore most app notifications.

That last point is the real design tension exposed in Section 7: **proactive systems need much higher precision than reactive ones**. A bad reactive answer just costs a follow-up question. A bad proactive suggestion costs attention the user didn't choose to spend — which is why the confidence score is shown alongside the suggestion rather than presenting every push notification with equal certainty.

My guess is the future isn't "no more typing" so much as a blend: pull interfaces stay for genuinely open-ended requests where the system has no way to guess what you want, while push interfaces take over the narrower, well-signaled, repetitive decisions — the ones where the system's confidence is actually high enough to earn the interruption.
